# TransRefNet — Google Colab Training Notebook

**TransRefNet: Transformer-Augmented Refinement Network for Biomedical Image Segmentation**

This notebook trains and evaluates TransRefNet and the ConvSegNet baseline on three biomedical segmentation datasets:
- **Kvasir-SEG** — polyp segmentation (1 000 images)
- **CVC-ClinicDB** — polyp segmentation (612 images)
- **ISIC 2018 Task 1** — skin lesion segmentation (2 594 images)

**Before running:** Runtime → Change runtime type → **T4 GPU**

---
### Notebook Sections
0. GPU check
1. Mount Google Drive
2. Clone repo & install dependencies
3. Kaggle credentials setup
4. Download datasets
5. Train — TransRefNet
6. Train — ConvSegNet baseline
7. Evaluate all checkpoints
8. Results summary table

---
## 0 — GPU Check

In [ ]:
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU')

---
## 1 — Mount Google Drive

Datasets, checkpoints, and results are stored on Drive so they persist across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Paths ────────────────────────────────────────────────────────────────────────────
BASE        = '/content/drive/MyDrive/TransRefNet'
DATA_ROOT   = f'{BASE}/data'
CKPT_DIR    = f'{BASE}/checkpoints'
RESULTS_DIR = f'{BASE}/results'
PRED_DIR    = f'{BASE}/predictions'

import os
for d in [BASE, DATA_ROOT, CKPT_DIR, RESULTS_DIR, PRED_DIR]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted. Base directory:', BASE)

---
## 2 — Clone Repository & Install Dependencies

In [ ]:
import os
from google.colab import userdata

# Store your GitHub Personal Access Token in Colab Secrets
# (the key icon in the left sidebar) under the name GITHUB_TOKEN.
# The token needs 'repo' scope for a private repository.
GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')

REPO_DIR = '/content/TransRefNet'
REPO_URL = f'https://{GITHUB_TOKEN}@github.com/ayokunleige/TransRefNet.git'

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# Install packages not pre-installed on Colab
!pip install -q albumentations ptflops kaggle
print('Dependencies installed.')

---
## 3 — Kaggle Credentials

The new Kaggle API uses a token string (starting with `KGAT_`) rather than a JSON file.

**Step 1:** Go to kaggle.com → Account → Settings → **Create New Token**  
**Step 2:** Copy the token string shown on the page  
**Step 3:** Add it as a Colab Secret (key icon in the left sidebar) named `KAGGLE_TOKEN`  
**Step 4:** Run the cell below

In [ ]:
import os
from google.colab import userdata

# Write the new-format Kaggle token to the expected location
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

token = userdata.get('KAGGLE_TOKEN')   # KGAT_... string from Colab Secrets
token_path = os.path.join(kaggle_dir, 'access_token')
with open(token_path, 'w') as f:
    f.write(token)
os.chmod(token_path, 0o600)

print(f'Kaggle token written to {token_path}')

---
## 4 — Download Datasets

Downloads are skipped automatically if the dataset folder already exists (idempotent).

In [ ]:
from download_data import download_all
download_all(DATA_ROOT)

In [ ]:
# Individual downloads if you prefer
# from download_data import download_kvasir, download_clinicdb, download_isic2018
# download_kvasir(DATA_ROOT)
# download_clinicdb(DATA_ROOT)
# download_isic2018(DATA_ROOT)

---
## 5 — Train TransRefNet (Proposed)

Trains TransRefNet on all three datasets sequentially.  
Checkpoints are saved to Drive and survive session restarts.

In [ ]:
DATASETS = ['kvasir', 'clinicdb', 'isic2018']

for dataset in DATASETS:
    print(f'\n{"="*60}')
    print(f'  Training TransRefNet on {dataset.upper()}')
    print(f'{"="*60}\n')
    !python train.py \
        --model      transrefnet \
        --dataset    {dataset} \
        --data_root  {DATA_ROOT} \
        --save_dir   {CKPT_DIR} \
        --epochs     300 \
        --batch_size 8 \
        --img_size   256 \
        --lr         1e-4 \
        --patience   50 \
        --seed       42

**Resume a specific dataset** (if a session restarted mid-training — the existing checkpoint is overwritten only when a better val-F1 is achieved):

In [ ]:
# Uncomment and edit to resume a single experiment
# !python train.py \
#     --model      transrefnet \
#     --dataset    isic2018 \
#     --data_root  {DATA_ROOT} \
#     --save_dir   {CKPT_DIR} \
#     --epochs     300 \
#     --batch_size 8

---
## 6 — Train ConvSegNet Baseline

Identical hyperparameters to ensure a fair comparison.

In [ ]:
DATASETS = ['kvasir', 'clinicdb', 'isic2018']

for dataset in DATASETS:
    print(f'\n{"="*60}')
    print(f'  Training ConvSegNet on {dataset.upper()}')
    print(f'{"="*60}\n')
    !python train.py \
        --model      convsegnet \
        --dataset    {dataset} \
        --data_root  {DATA_ROOT} \
        --save_dir   {CKPT_DIR} \
        --epochs     300 \
        --batch_size 8 \
        --img_size   256 \
        --lr         1e-4 \
        --patience   50 \
        --seed       42

---
## 7 — Evaluate All Checkpoints

Runs `test.py` on the held-out test split for each model × dataset combination.  
Per-sample CSV and aggregate metrics are saved to Drive.

In [ ]:
import os

MODELS   = ['transrefnet', 'convsegnet']
DATASETS = ['kvasir', 'clinicdb', 'isic2018']

for model in MODELS:
    for dataset in DATASETS:
        ckpt = os.path.join(CKPT_DIR, f'{model}_{dataset}', 'best_checkpoint.pth')
        if not os.path.exists(ckpt):
            print(f'[SKIP] Checkpoint not found: {ckpt}')
            continue
        print(f'\nEvaluating {model.upper()} on {dataset.upper()}...')
        !python test.py \
            --checkpoint  {ckpt} \
            --dataset     {dataset} \
            --data_root   {DATA_ROOT} \
            --results_dir {RESULTS_DIR}/{model}_{dataset} \
            --save_preds  {PRED_DIR}/{model}_{dataset} \
            --batch_size  8

---
## 8 — Results Summary Table

Loads all `test_summary.txt` files and builds a comparison table.

In [ ]:
import os, re
import pandas as pd

MODELS   = ['transrefnet', 'convsegnet']
DATASETS = ['kvasir', 'clinicdb', 'isic2018']

rows = []
for model in MODELS:
    for dataset in DATASETS:
        summary_path = os.path.join(RESULTS_DIR, f'{model}_{dataset}', 'test_summary.txt')
        if not os.path.exists(summary_path):
            continue
        txt = open(summary_path).read()
        def extract(key):
            m = re.search(rf'{key}\s*:\s*([\d.]+)', txt)
            return float(m.group(1)) if m else None
        rows.append({
            'Model':     model,
            'Dataset':   dataset.upper(),
            'Jaccard':   extract('Jaccard'),
            'F1 (Dice)': extract('F1'),
            'Recall':    extract('Recall'),
            'Precision': extract('Precision'),
        })

if rows:
    df = pd.DataFrame(rows).set_index(['Model', 'Dataset'])
    pd.set_option('display.float_format', '{:.4f}'.format)
    display(df)
    # Save as CSV for the paper
    out_csv = os.path.join(RESULTS_DIR, 'all_results.csv')
    df.to_csv(out_csv)
    print(f'\nSaved to: {out_csv}')
else:
    print('No results found yet. Run Section 7 first.')